# Logistic Regression - Complete Tutorial

## 🎯 Goal
Predict if a student will **pass or fail** an exam based on hours studied and practice problems

## 📚 What You'll Learn
- The sigmoid function and probability
- How to train a Logistic Regression model
- How to interpret coefficients (log-odds)
- Decision boundaries and thresholds
- All evaluation metrics (accuracy, precision, recall, F1, ROC-AUC)
- Confusion matrix interpretation

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

np.random.seed(42)
sns.set_style('whitegrid')

print('✅ Libraries imported!')

## Step 2: Visualize the Sigmoid Function

First, let's understand the magic behind logistic regression!

In [ ]:
def sigmoid(z):
    """The sigmoid function - converts any number to 0-1"""
    return 1 / (1 + np.exp(-z))

# Test sigmoid with different values
test_values = [-5, -2, -1, 0, 1, 2, 5]
print('Sigmoid Examples:')
print('=' * 40)
for z in test_values:
    print(f'  σ({z:3d}) = {sigmoid(z):.4f} ({sigmoid(z)*100:.1f}%)')

# Plot the sigmoid
z = np.linspace(-10, 10, 100)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z, sigmoid(z), linewidth=3, color='blue')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5, label='z = 0')
ax.scatter([0], [0.5], color='red', s=100, zorder=5)
ax.set_xlabel('Input (z)', fontsize=12)
ax.set_ylabel('Output σ(z)', fontsize=12)
ax.set_title('The Sigmoid Function: Squishes Any Number to 0-1', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Create Sample Data

Generate realistic student data.

In [ ]:
# Generate student data
n_students = 200
hours = np.random.uniform(0, 12, n_students)
practice = np.random.uniform(0, 100, n_students)

# True relationship (with noise)
score = -5 + 0.6*hours + 0.05*practice + np.random.normal(0, 1, n_students)
passed = (score > 0).astype(int)

df = pd.DataFrame({
    'hours': hours.round(1),
    'practice': practice.round(0).astype(int),
    'passed': passed
})

print('📊 Student Dataset:')
print(df.head(10))
print(f'\nTotal students: {len(df)}')
print(f'Passed: {df["passed"].sum()} ({df["passed"].mean()*100:.1f}%)')
print(f'Failed: {(1-df["passed"]).sum()} ({(1-df["passed"]).mean()*100:.1f}%)')

## Step 4: Visualize the Data

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(df[df['passed']==0]['hours'], df[df['passed']==0]['practice'],
           c='red', label='Failed', s=80, edgecolors='black', alpha=0.7)
ax.scatter(df[df['passed']==1]['hours'], df[df['passed']==1]['practice'],
           c='green', label='Passed', s=80, edgecolors='black', alpha=0.7)

ax.set_xlabel('Hours Studied', fontsize=12)
ax.set_ylabel('Practice Problems', fontsize=12)
ax.set_title('Student Performance: Pass vs Fail', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Notice: There seems to be a pattern!')
print('More hours + more practice = more likely to pass')

## Step 5: Prepare Data and Train Model

In [ ]:
# Split data
X = df[['hours', 'practice']]
y = df['passed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (good practice!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print('🤖 Model trained!')
print(f'\n📋 Model Parameters:')
print(f'  Intercept (β₀): {model.intercept_[0]:.4f}')
print(f'  Hours coef:     {model.coef_[0][0]:.4f}')
print(f'  Practice coef:  {model.coef_[0][1]:.4f}')

print('\n💡 Interpretation:')
print('  - Both coefficients are POSITIVE')
print('  - More hours studied → higher P(pass) ✅')
print('  - More practice → higher P(pass) ✅')

## Step 6: Make Predictions and Get Probabilities

In [ ]:
# Get predictions and probabilities
predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)

# Show first 10 predictions
results_df = pd.DataFrame({
    'Hours': X_test['hours'].values[:10],
    'Practice': X_test['practice'].values[:10],
    'P(Fail)': probabilities[:10, 0].round(3),
    'P(Pass)': probabilities[:10, 1].round(3),
    'Predicted': predictions[:10],
    'Actual': y_test.values[:10]
})

results_df['Correct'] = results_df['Predicted'] == results_df['Actual']
print('🔮 First 10 Predictions:')
print(results_df.to_string(index=False))

## Step 7: Evaluate the Model

In [ ]:
# Calculate all metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print('📊 Evaluation Metrics:')
print('=' * 40)
print(f'  Accuracy:  {accuracy:.2%}')
print(f'  Precision: {precision:.2%}')
print(f'  Recall:    {recall:.2%}')
print(f'  F1 Score:  {f1:.2%}')

print('\n📋 Full Classification Report:')
print(classification_report(y_test, predictions, target_names=['Failed', 'Passed']))

## Step 8: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Failed', 'Passed'],
            yticklabels=['Failed', 'Passed'],
            cbar_kws={'label': 'Count'},
            ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

TN, FP = cm[0]
FN, TP = cm[1]

print(f'\n📊 Breakdown:')
print(f'  True Negatives (correctly failed): {TN}')
print(f'  False Positives (incorrectly passed): {FP}')
print(f'  False Negatives (incorrectly failed): {FN}')
print(f'  True Positives (correctly passed): {TP}')

## Step 9: Visualize the Decision Boundary

See where the model draws the line between Pass and Fail!

In [ ]:
# Create mesh
x_min, x_max = X['hours'].min() - 1, X['hours'].max() + 1
y_min, y_max = X['practice'].min() - 5, X['practice'].max() + 5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Scale and predict
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = scaler.transform(grid)
Z = model.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Probability contour
contour = ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.5)
plt.colorbar(contour, label='P(Pass)')

# Decision boundary at 0.5
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)

# Data points
ax.scatter(df[df['passed']==0]['hours'], df[df['passed']==0]['practice'],
           c='red', label='Failed', s=80, edgecolors='black', alpha=0.8)
ax.scatter(df[df['passed']==1]['hours'], df[df['passed']==1]['practice'],
           c='green', label='Passed', s=80, edgecolors='black', alpha=0.8)

ax.set_xlabel('Hours Studied', fontsize=12)
ax.set_ylabel('Practice Problems', fontsize=12)
ax.set_title('Logistic Regression Decision Boundary', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='lower right')
plt.tight_layout()
plt.show()

print('\n💡 The black line is the decision boundary (P=0.5)')
print('   Above the line → Pass | Below → Fail')
print('   Color shows confidence (greener = more likely to pass)')

## Step 10: ROC Curve and AUC

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, probabilities[:, 1])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='blue', linewidth=3, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.2, color='blue')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 AUC Score: {roc_auc:.3f}')
if roc_auc >= 0.9:
    print('   ✅ Excellent classifier!')
elif roc_auc >= 0.8:
    print('   ✅ Good classifier!')
elif roc_auc >= 0.7:
    print('   ✅ Decent classifier')
else:
    print('   ⚠️ Needs improvement')

## Step 11: Try Custom Thresholds

What if we change the decision threshold from 0.5?

In [ ]:
thresholds_to_try = [0.3, 0.5, 0.7]
results = []

for thresh in thresholds_to_try:
    pred_thresh = (probabilities[:, 1] >= thresh).astype(int)
    results.append({
        'Threshold': thresh,
        'Accuracy': accuracy_score(y_test, pred_thresh),
        'Precision': precision_score(y_test, pred_thresh, zero_division=0),
        'Recall': recall_score(y_test, pred_thresh, zero_division=0),
        'F1': f1_score(y_test, pred_thresh, zero_division=0)
    })

thresh_df = pd.DataFrame(results)
print('🎚️ Threshold Comparison:')
print(thresh_df.round(3).to_string(index=False))

print('\n💡 Notice the trade-off:')
print('   Lower threshold (0.3) → Higher recall, lower precision')
print('   Higher threshold (0.7) → Higher precision, lower recall')
print('   Choose based on your problem!')

## Step 12: Predict for New Students

In [ ]:
def predict_student(hours, practice):
    """Predict if a student will pass"""
    features = np.array([[hours, practice]])
    features_scaled = scaler.transform(features)
    
    prob = model.predict_proba(features_scaled)[0, 1]
    pred = model.predict(features_scaled)[0]
    
    print(f'\n👤 Student Profile:')
    print(f'   Hours studied: {hours}')
    print(f'   Practice problems: {practice}')
    print(f'\n   📊 P(Pass) = {prob:.1%}')
    print(f'   🎯 Prediction: {"PASS ✅" if pred == 1 else "FAIL ❌"}')
    
    if prob > 0.8:
        print(f'   💪 Highly confident pass')
    elif prob > 0.6:
        print(f'   📚 Likely to pass, keep working!')
    elif prob > 0.4:
        print(f'   ⚠️ Borderline, study more!')
    else:
        print(f'   📖 Need more preparation!')

# Test cases
predict_student(hours=2, practice=20)   # Low effort
predict_student(hours=5, practice=50)   # Medium effort
predict_student(hours=10, practice=80)  # High effort
predict_student(hours=8, practice=30)   # Mixed

## 🎓 Summary

### What We Built:
1. ✅ Visualized the sigmoid function
2. ✅ Created and explored data
3. ✅ Trained a Logistic Regression model
4. ✅ Made predictions with probabilities
5. ✅ Evaluated with multiple metrics
6. ✅ Visualized confusion matrix and decision boundary
7. ✅ Plotted ROC curve and calculated AUC
8. ✅ Experimented with custom thresholds
9. ✅ Built a prediction function

### Key Takeaways:
- **Sigmoid** converts any number to a probability (0-1)
- **Threshold** of 0.5 is default but can be adjusted
- **Multiple metrics** matter (accuracy alone can mislead)
- **Decision boundary** shows where model is uncertain
- **ROC-AUC** is a great single number to compare models

### Try These Exercises:
1. Use the iris dataset (3-class classification)
2. Try with imbalanced data (class_weight='balanced')
3. Add regularization (penalty='l1' or 'l2')
4. Build a binary classifier on real data (Kaggle)

**Happy Classifying! 🎯✨**